# GetAround - Analyse & Deployment

**Objectif** : analyser les retards de checkout et leur impact, puis construire un modele de pricing deploye via API.

- **Partie 1** : EDA sur les retards (delay analysis)
- **Partie 2** : ML pour le pricing (regression)

## Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
# Partie 1 : Analyse des retards

## 1.1 Chargement des donnees

In [2]:
df_delay = pd.read_excel('data/get_around_delay_analysis.xlsx')
df_delay.shape

(21310, 7)

In [3]:
df_delay.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [4]:
df_delay.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21310 entries, 0 to 21309
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   rental_id                                   21310 non-null  int64  
 1   car_id                                      21310 non-null  int64  
 2   checkin_type                                21310 non-null  object 
 3   state                                       21310 non-null  object 
 4   delay_at_checkout_in_minutes                16346 non-null  float64
 5   previous_ended_rental_id                    1841 non-null   float64
 6   time_delta_with_previous_rental_in_minutes  1841 non-null   float64
dtypes: float64(3), int64(2), object(2)
memory usage: 1.1+ MB


In [5]:
df_delay.describe()

,rental_id,car_id,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
count,21310.000000,21310.000000,16346.000000,1841.000000,1841.000000
mean,549712.880338,350030.603426,59.701517,550127.411733,279.288430
std,13863.446964,58206.249765,1002.561635,13184.023111,254.594486
min,504806.000000,159250.000000,-22433.000000,505628.000000,0.000000
25%,540613.250000,317639.000000,-36.000000,540896.000000,60.000000
50%,550350.000000,368717.000000,9.000000,550567.000000,180.000000
75%,560468.500000,394928.000000,67.000000,560823.000000,540.000000
max,576401.000000,417675.000000,71084.000000,575053.000000,720.000000


## 1.2 Donnees manquantes

In [6]:
df_delay.isnull().sum()

rental_id                                         0
car_id                                            0
checkin_type                                      0
state                                             0
delay_at_checkout_in_minutes                   4964
previous_ended_rental_id                      19469
time_delta_with_previous_rental_in_minutes    19469
dtype: int64

In [7]:
# Analyse des valeurs manquantes :
# - delay_at_checkout : NaN pour les 3 265 locations annulees (pas de checkout)
#   MAIS aussi pour ~1 700 locations terminees dont le delay n'a pas ete enregistre
#   (4 964 NaN au total - 3 265 canceled = 1 699 ended sans donnee)
# - previous_ended_rental_id et time_delta : NaN quand pas de location precedente sur la meme voiture (91.4%)

print(f"Locations sans checkout (canceled): {df_delay[df_delay['state']=='canceled'].shape[0]}")
print(f"Locations terminees sans delay: {df_delay[(df_delay['state']=='ended') & df_delay['delay_at_checkout_in_minutes'].isna()].shape[0]}")
print(f"Locations sans precedent: {df_delay['previous_ended_rental_id'].isna().sum()}")

Locations sans checkout (canceled): 3265
Locations terminees sans delay: 1700
Locations sans precedent: 19469


## 1.3 Vue d'ensemble

In [8]:
total = len(df_delay)
ended = df_delay[df_delay['state'] == 'ended']
canceled = df_delay[df_delay['state'] == 'canceled']

print(f"Total: {total}")
print(f"Terminees: {len(ended)} ({len(ended)/total*100:.1f}%)")
print(f"Annulees: {len(canceled)} ({len(canceled)/total*100:.1f}%)")

Total: 21310
Terminees: 18045 (84.7%)
Annulees: 3265 (15.3%)


In [9]:
fig = px.pie(df_delay, names='checkin_type', title='Repartition par type de checkin')
fig.show()

In [10]:
fig = px.pie(df_delay, names='state', title='Repartition par statut')
fig.show()

## 1.4 Analyse des retards au checkout

In [11]:
delays = ended['delay_at_checkout_in_minutes'].dropna()
late = delays[delays > 0]
on_time = delays[delays <= 0]

print(f"Locations avec delay renseigne: {len(delays)}")
print(f"En retard (>0 min): {len(late)} ({len(late)/len(delays)*100:.1f}%)")
print(f"A l'heure ou en avance: {len(on_time)} ({len(on_time)/len(delays)*100:.1f}%)")
print(f"Retard moyen (quand retard): {late.mean():.0f} min ({late.mean()/60:.1f}h)")
print(f"Retard median (quand retard): {late.median():.0f} min")

Locations avec delay renseigne: 16345
En retard (>0 min): 9404 (57.5%)
A l'heure ou en avance: 6941 (42.5%)
Retard moyen (quand retard): 202 min (3.4h)
Retard median (quand retard): 53 min


In [12]:
# Distribution des retards (clippe pour lisibilite)
fig = px.histogram(
    delays.clip(-200, 500), 
    nbins=100, 
    title='Distribution des retards au checkout (clippe -200 a 500 min)',
    labels={'value': 'Retard (minutes)', 'count': 'Nombre'}
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Heure prevue')
fig.show()

In [13]:
# Retards par type de checkin
delay_by_type = ended.groupby('checkin_type')['delay_at_checkout_in_minutes'].agg(['mean', 'median', 'count'])
delay_by_type.columns = ['Retard moyen', 'Retard median', 'Nb locations']
delay_by_type

,Retard moyen,Retard median,Nb locations
checkin_type,,,
connect,-43.627278,-9.0,3402
mobile,88.215174,14.0,12943


In [14]:
fig = px.box(
    ended, x='checkin_type', y='delay_at_checkout_in_minutes',
    title='Distribution des retards par type de checkin',
    range_y=[-200, 500]
)
fig.show()

**Observations** : les locations **Connect** sont en moyenne rendues **en avance** (median -9 min, moyenne -44 min), tandis que les locations **Mobile** sont en moyenne rendues **en retard** (median +14 min, moyenne +88 min). Cela s'explique par le checkin sans contact des voitures Connect qui impose moins de friction au retour. Malgre cela, les voitures Connect restent sensibles aux retards car le conducteur suivant ne peut pas contacter le precedent en cas de probleme.

## 1.5 Impact sur la location suivante

In [15]:
# Locations qui ont une location precedente (potentiel conflit)
with_prev = df_delay[df_delay['previous_ended_rental_id'].notna()].copy()
print(f"Locations avec location precedente: {len(with_prev)} ({len(with_prev)/total*100:.1f}%)")

# Recuperer le retard de la location precedente
prev_delays = df_delay.set_index('rental_id')['delay_at_checkout_in_minutes']
with_prev['prev_delay'] = with_prev['previous_ended_rental_id'].map(prev_delays)

Locations avec location precedente: 1841 (8.6%)


In [16]:
# Cas problematiques : retard precedent > buffer disponible
problematic = with_prev[with_prev['prev_delay'] > with_prev['time_delta_with_previous_rental_in_minutes']]
print(f"Cas problematiques (retard depasse le buffer): {len(problematic)}")
print(f"Soit {len(problematic)/len(with_prev)*100:.1f}% des locations consecutives")

# Annulations parmi les cas problematiques
prob_canceled = problematic[problematic['state'] == 'canceled']
print(f"Dont annulations: {len(prob_canceled)} ({len(prob_canceled)/len(problematic)*100:.1f}%)")

Cas problematiques (retard depasse le buffer): 218
Soit 11.8% des locations consecutives
Dont annulations: 37 (17.0%)


In [17]:
fig = px.histogram(
    with_prev['time_delta_with_previous_rental_in_minutes'],
    nbins=50,
    title='Distribution du delta entre locations consecutives',
    labels={'value': 'Delta (minutes)', 'count': 'Nombre'}
)
fig.show()

## 1.6 Simulation des seuils (threshold)

In [18]:
results = []
for threshold in [0, 15, 30, 60, 120, 180, 240, 360, 720]:
    for scope in ['all', 'connect']:
        subset = with_prev if scope == 'all' else with_prev[with_prev['checkin_type'] == 'connect']
        prob_sub = problematic if scope == 'all' else problematic[problematic['checkin_type'] == 'connect']
        
        if threshold == 0:
            affected = 0
            solved = 0
        else:
            affected = (subset['time_delta_with_previous_rental_in_minutes'] < threshold).sum()
            solved = (prob_sub['time_delta_with_previous_rental_in_minutes'] + threshold >= prob_sub['prev_delay']).sum()
        
        results.append({
            'threshold_min': threshold,
            'scope': scope,
            'locations_affectees': affected,
            'pct_affectees': round(affected / len(subset) * 100, 1) if len(subset) > 0 else 0,
            'cas_resolus': solved,
            'pct_resolus': round(solved / len(prob_sub) * 100, 1) if len(prob_sub) > 0 else 0
        })

df_sim = pd.DataFrame(results)
df_sim

,threshold_min,scope,locations_affectees,pct_affectees,cas_resolus,pct_resolus
0,0,all,0,0.0,0,0.0
1,0,connect,0,0.0,0,0.0
2,15,all,279,15.2,75,34.4
3,15,connect,131,16.1,22,31.9
4,30,all,279,15.2,115,52.8
5,30,connect,131,16.1,32,46.4
6,60,all,401,21.8,152,69.7
7,60,connect,181,22.3,49,71.0
8,120,all,666,36.2,177,81.2
9,120,connect,295,36.3,58,84.1


In [19]:
# Tradeoff: cas resolus vs locations bloquees
fig = make_subplots(specs=[[{'secondary_y': True}]])

for scope in ['all', 'connect']:
    sub = df_sim[df_sim['scope'] == scope]
    fig.add_trace(
        go.Scatter(x=sub['threshold_min'], y=sub['pct_resolus'], 
                   name=f'Cas resolus ({scope})', mode='lines+markers'),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=sub['threshold_min'], y=sub['pct_affectees'], 
                   name=f'Locations bloquees ({scope})', mode='lines+markers', line=dict(dash='dash')),
        secondary_y=True
    )

fig.update_layout(title='Tradeoff: cas resolus vs locations bloquees selon le seuil')
fig.update_xaxes(title_text='Seuil (minutes)')
fig.update_yaxes(title_text='% cas problematiques resolus', secondary_y=False)
fig.update_yaxes(title_text='% locations bloquees', secondary_y=True)
fig.show()

## 1.7 Recommandation

**Seuil recommande : 120 minutes** (scope : connect uniquement)

- Resout ~84% des cas problematiques pour les voitures Connect
- Ne bloque que ~36% des locations consecutives Connect
- Les voitures Connect representent 20% du parc mais ont un checkin sans contact, donc plus sensibles aux retards
- Un seuil de 120 min est un bon compromis entre protection des utilisateurs et perte de revenus

---
# Partie 2 : Pricing - Machine Learning

## 2.1 Chargement et exploration

In [20]:
df_price = pd.read_csv('data/get_around_pricing_project.csv', index_col=0)
df_price.shape

(4843, 14)

In [21]:
df_price.head()

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183


In [22]:
df_price.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4843 entries, 0 to 4842
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   model_key                  4843 non-null   object
 1   mileage                    4843 non-null   int64 
 2   engine_power               4843 non-null   int64 
 3   fuel                       4843 non-null   object
 4   paint_color                4843 non-null   object
 5   car_type                   4843 non-null   object
 6   private_parking_available  4843 non-null   bool  
 7   has_gps                    4843 non-null   bool  
 8   has_air_conditioning       4843 non-null   bool  
 9   automatic_car              4843 non-null   bool  
 10  has_getaround_connect      4843 non-null   bool  
 11  has_speed_regulator        4843 non-null   bool  
 12  winter_tires               4843 non-null   bool  
 13  rental_price_per_day       4843 non-null   int64 
dtypes: bool(7), i

In [23]:
df_price.describe()

,mileage,engine_power,rental_price_per_day
count,4.843000e+03,4843.00000,4843.000000
mean,1.409628e+05,128.98823,121.214536
std,6.019674e+04,38.99336,33.568268
min,-6.400000e+01,0.00000,10.000000
25%,1.029135e+05,100.00000,104.000000
50%,1.410800e+05,120.00000,119.000000
75%,1.751955e+05,135.00000,136.000000
max,1.000376e+06,423.00000,422.000000


In [24]:
df_price.isnull().sum()

model_key                    0
mileage                      0
engine_power                 0
fuel                         0
paint_color                  0
car_type                     0
private_parking_available    0
has_gps                      0
has_air_conditioning         0
automatic_car                0
has_getaround_connect        0
has_speed_regulator          0
winter_tires                 0
rental_price_per_day         0
dtype: int64

## 2.2 Analyse exploratoire

In [25]:
fig = px.histogram(df_price, x='rental_price_per_day', nbins=50, title='Distribution du prix journalier')
fig.show()

In [26]:
fig = px.box(df_price, x='car_type', y='rental_price_per_day', title='Prix par type de vehicule')
fig.show()

In [27]:
fig = px.scatter(df_price, x='engine_power', y='rental_price_per_day', 
                color='fuel', title='Prix vs puissance moteur')
fig.show()

In [28]:
fig = px.scatter(df_price, x='mileage', y='rental_price_per_day',
                color='car_type', title='Prix vs kilometrage')
fig.show()

In [29]:
# Correlation des features numeriques et booleennes
bool_cols = df_price.select_dtypes(include='bool').columns
df_corr = df_price.copy()
for col in bool_cols:
    df_corr[col] = df_corr[col].astype(int)
    
corr = df_corr.select_dtypes(include=[np.number]).corr()['rental_price_per_day'].drop('rental_price_per_day').sort_values(ascending=False)
print('Correlations avec le prix:')
print(corr)

Correlations avec le prix:
engine_power                 0.625645
automatic_car                0.419761
has_getaround_connect        0.318486
has_gps                      0.310889
private_parking_available    0.281358
has_air_conditioning         0.245386
has_speed_regulator          0.227547
winter_tires                 0.018277
mileage                     -0.448912
Name: rental_price_per_day, dtype: float64


In [30]:
fig = px.bar(
    x=corr.values, y=corr.index, orientation='h',
    title='Correlation des features avec le prix',
    labels={'x': 'Correlation', 'y': 'Feature'}
)
fig.show()

**Observations correlations** :
- **engine_power** (+0.63) : la puissance moteur est le meilleur predicteur du prix, ce qui est logique (voitures puissantes = gamme superieure)
- **mileage** (-0.45) : le kilometrage fait baisser le prix (usure, decote)
- **automatic_car** (+0.42) : les automatiques sont en general plus cheres
- **has_getaround_connect** (+0.32), **has_gps** (+0.31) : les equipements technologiques augmentent le prix
- **winter_tires** (+0.02) : correlation quasi nulle, les pneus hiver n'influencent pas le prix

## 2.3 Preprocessing et modelisation

In [31]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

In [32]:
target = 'rental_price_per_day'
X = df_price.drop(columns=[target])
y = df_price[target]

cat_cols = [c for c in X.columns if X[c].dtype.name in ('object', 'str', 'string', 'string[python]')]
bool_cols = [c for c in X.columns if X[c].dtype.name == 'bool']
num_cols = [c for c in X.columns if c not in cat_cols and c not in bool_cols]

for col in bool_cols:
    X[col] = X[col].astype(int)

print(f'Categoriques: {cat_cols}')
print(f'Booleens: {bool_cols}')
print(f'Numeriques: {num_cols}')

Categoriques: ['model_key', 'fuel', 'paint_color', 'car_type']
Booleens: ['private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']
Numeriques: ['mileage', 'engine_power']


In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (3874, 13), Test: (969, 13)


In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
        ('bool', 'passthrough', bool_cols)
    ]
)

In [35]:
# Linear Regression
lr_pipe = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)

print("=== Linear Regression ===")
print(f"R2:   {r2_score(y_test, lr_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, lr_pred):.2f} EUR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, lr_pred)):.2f} EUR")

=== Linear Regression ===
R2:   0.6937
MAE:  12.12 EUR
RMSE: 17.96 EUR


In [36]:
# Gradient Boosting
gb_pipe = Pipeline([
    ('preprocessor', preprocessor), 
    ('model', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
])
gb_pipe.fit(X_train, y_train)
gb_pred = gb_pipe.predict(X_test)

print("=== Gradient Boosting ===")
print(f"R2:   {r2_score(y_test, gb_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, gb_pred):.2f} EUR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, gb_pred)):.2f} EUR")

=== Gradient Boosting ===
R2:   0.7504
MAE:  10.29 EUR
RMSE: 16.22 EUR


In [37]:
# Cross-validation
# Note : les warnings "unknown categories" sont attendus car certaines categories rares
# (model_key, car_type) n'apparaissent pas dans tous les folds. handle_unknown='ignore' les gere.
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
    cv_scores = cross_val_score(gb_pipe, X, y, cv=5, scoring='r2')
print(f'CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Scores par fold: {[round(s, 4) for s in cv_scores]}')

CV R2: 0.6932 (+/- 0.0700)
Scores par fold: [np.float64(0.5847), np.float64(0.6716), np.float64(0.7745), np.float64(0.7649), np.float64(0.6704)]


**Analyse CV vs Test** : le R2 en cross-validation (0.693) est inferieur au R2 sur le test set (0.750), ce qui indique un **leger overfitting** du Gradient Boosting. L'ecart reste modere (~0.06 points) et la variance entre folds (0.58 a 0.77) reflete l'heterogeneite des donnees. Le modele reste fiable pour une utilisation en production avec une erreur moyenne de ~10 EUR.

## 2.4 MLflow tracking

In [38]:
import mlflow
import mlflow.sklearn
import os
import warnings
import logging

# Suppression des warnings informatifs (fichier backend + pickle format)
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")
logging.getLogger("mlflow.sklearn").setLevel(logging.ERROR)

mlflow.set_tracking_uri("file:///" + os.path.abspath("mlruns").replace("\\", "/"))
mlflow.set_experiment("getaround_pricing")

# Log du modele Linear Regression
with mlflow.start_run(run_name="linear_regression"):
    lr_pipe_mlf = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
    lr_pipe_mlf.fit(X_train, y_train)
    lr_pred_mlf = lr_pipe_mlf.predict(X_test)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_metric("r2", r2_score(y_test, lr_pred_mlf))
    mlflow.log_metric("mae", mean_absolute_error(y_test, lr_pred_mlf))
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, lr_pred_mlf)))
    mlflow.sklearn.log_model(lr_pipe_mlf, name="model")

# Log du modele Gradient Boosting
with mlflow.start_run(run_name="gradient_boosting"):
    gb_pipe_mlf = Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
    ])
    gb_pipe_mlf.fit(X_train, y_train)
    gb_pred_mlf = gb_pipe_mlf.predict(X_test)
    
    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("r2", r2_score(y_test, gb_pred_mlf))
    mlflow.log_metric("mae", mean_absolute_error(y_test, gb_pred_mlf))
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, gb_pred_mlf)))
    mlflow.sklearn.log_model(gb_pipe_mlf, name="model")

print("Runs MLflow enregistres")

Runs MLflow enregistres


## 2.5 Feature importance

In [39]:
feature_names = (num_cols + 
    list(gb_pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()) +
    bool_cols)

importances = gb_pipe.named_steps['model'].feature_importances_
fi = pd.Series(importances, index=feature_names).sort_values(ascending=True)

fig = px.bar(x=fi.values, y=fi.index, orientation='h', 
             title='Feature importance (Gradient Boosting)',
             labels={'x': 'Importance', 'y': 'Feature'})
fig.update_layout(height=600)
fig.show()

In [40]:
# Predictions vs valeurs reelles
fig = px.scatter(x=y_test, y=gb_pred, 
                 labels={'x': 'Prix reel (EUR)', 'y': 'Prix predit (EUR)'},
                 title='Predictions vs valeurs reelles (Gradient Boosting)')
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
                         mode='lines', line=dict(color='red', dash='dash'), name='y=x'))
fig.show()

## 2.6 Export du modele

In [41]:
joblib.dump(gb_pipe, 'api/model.joblib')
print('Modele exporte dans api/model.joblib')

Modele exporte dans api/model.joblib


In [42]:
# Verification
loaded_model = joblib.load('api/model.joblib')
sample = X_test.iloc[:3]
preds = loaded_model.predict(sample)
print("Test de prediction:")
for i in range(3):
    print(f"  Predit: {preds[i]:.0f} EUR | Reel: {y_test.iloc[i]} EUR")

Test de prediction:
  Predit: 138 EUR | Reel: 152 EUR
  Predit: 113 EUR | Reel: 120 EUR
  Predit: 154 EUR | Reel: 158 EUR


---
## Synthese

**Analyse des retards :**
- 57.5% des locations (avec donnees de checkout) se terminent en retard, avec un retard median de 53 min
- Les voitures Connect sont en moyenne rendues en avance (-44 min) vs Mobile en retard (+88 min)
- 218 cas problematiques identifies sur 1 841 locations consecutives (11.8%), dont 37 annulations (17%)
- **Recommandation** : seuil de 120 min, scope Connect → resout 84% des conflits, bloque 36% des locations consecutives Connect

**Pricing ML :**
- Gradient Boosting retenu : R2=0.75 (test), MAE=10.29 EUR, RMSE=16.22 EUR
- CV R2=0.69 (+/- 0.07) → leger overfitting mais performances stables
- Features les plus correlees au prix : puissance moteur (+0.63), kilometrage (-0.45), automatique (+0.42), Connect (+0.32), GPS (+0.31)
- Modele exporte en pipeline sklearn et deploye via API FastAPI sur HuggingFace Spaces